# Part 1.12 Experiments

After this notebook you can run the workflow over a dataset with two models and compare
the results side by side in LangSmith.

In [ ]:
from kea.config import MODEL, setup

setup("kea-claims-local")

In [ ]:
# Each run of the experiments below makes real model calls and new experiments.
RUN_EXPERIMENTS = True

## See it

In LangSmith, open the `kea-claims` dataset, go to **Experiments**, select the two
experiments, and click **Compare**. The four feedback columns sit side by side. Open a
row that differs to see both traces.

<img src="../../assets/screenshots/p1-12-compare.png" alt="the Compare view for the two models: the four feedback columns side by side, with one regression row expanded." width="900">

## Build it

An experiment runs every example through a **target** and scores it. Our target calls
the same `builder` you used all along, with an in-memory checkpointer and Store, and
approves any human review itself so every claim finishes. Run it for two models.

In [ ]:
from langsmith import Client

from kea.config import ALT_MODEL
from kea.evals.claims import (
    DATASET_NAME,
    EVALUATORS,
    build_dataset,
    make_target,
    mean_scores,
)

client = Client()
if not client.has_dataset(dataset_name=DATASET_NAME):
    build_dataset(client)
examples = list(client.list_examples(dataset_name=DATASET_NAME, include_attachments=True))


def run(model: str):
    return client.evaluate(
        make_target(model),
        data=examples,
        evaluators=EVALUATORS,
        experiment_prefix=f"claims-{model.split('/')[-1]}",
        metadata={"model": model},
        max_concurrency=4,
    )

In [ ]:
if RUN_EXPERIMENTS:
    for model in (MODEL, ALT_MODEL):
        results = run(model)
        print(model, results.experiment_name)
        print({key: round(score, 2) for key, score in mean_scores(results).items()})
else:
    print("RUN_EXPERIMENTS is False: skipping the experiments.")

Now open the dataset in LangSmith and compare the two experiments, as in the screenshot
above. Look at `damage_type_correct` and `letter_quality`, then open a row that differs.

## Under the hood

- The model is a **context** value, `Context(model=...)`, not part of the graph.
  `make_target(model)` passes it per run, so the same graph code is compared across
  models with nothing else changed.
- `evaluators=EVALUATORS` are the four from the last notebook. Each example's scores
  show up as feedback on its trace.
- `max_concurrency=4` runs examples in parallel. Each example gets a fresh checkpointer
  and Store, so runs don't leak state into each other.
- Compare on the `ambiguous` and `guardrail` splits first. That is where models
  disagree.

**Swapping the model provider.** Models come from `init_chat_model`, so a different
provider is a different string. For example, Claude on Amazon Bedrock (not run here):

```python
from langchain.chat_models import init_chat_model

llm = init_chat_model("bedrock_converse:anthropic.claude-...")
```

Or set it once for everything, including the experiments:

```sh
export KEA_MODEL="bedrock_converse:anthropic.claude-..."
```

Docs: [Evaluation](https://docs.langchain.com/langsmith/evaluation).

## Recap

- An experiment is a target, a dataset, and evaluators, run once per model or change.
- Pass the model through `Context` so only one thing changes between experiments.
- Compare view turns "it looks right" into scores you can argue about.
- That is the workflow, built and tested. Part 2 hands the decisions to a model.

**Next:** [Part 2.1 The rebooking agent in Studio](../part2-deep-agents/01_rebooking_agent_in_studio.ipynb)